In [ ]:
#imports packages and loads data
#this data represents french insurance quote requests collected by an insurnace broker based in France
import pandas as pd
pd.set_option('display.max_columns', None)
df = pd.read_csv("../data/prospects.csv", dtype={'zipcode': str, 'num_tel': str})
print(df.head())

In [ ]:
#renames columns titles to English
df.rename(columns={
    'nom': 'last_name',
    'prenom': 'first_name',
    'civilite': 'title',
    'adresse': 'address',
    'ville': 'city',
    'zipcode': 'zip_code',
    'num_tel': 'phone_number',
    'email': 'email',
    'regime': 'social_security_regime',
    'date_naiss': 'birth_date',
    'date_effect': 'effective_date',
    'conjointbirthdate': 'spouse_birth_date',
    'nbrenfants': 'num_children',
    'source': 'source',
    'birthdatechild1': 'child1_birth_date',
    'birthdatechild2': 'child2_birth_date',
    'birthdatechild3': 'child3_birth_date',
    'birthdatechild4': 'child4_birth_date',
    'birthdatechild5': 'child5_birth_date',
    'dcr': 'submission_date',
    'soin_medical': 'medical_care',
    'hospitalisation': 'hospitalization',
    'optique': 'optical',
    'dentaire': 'dental'
}, inplace=True)

In [ ]:
#explores data
print(df.info())
print(df.describe(include='all'))

In [ ]:
#Explores and cleans data

#checks for empty strings
df.apply(lambda x: (x == '').sum())
df.apply(lambda x: (x.str.strip() == '').sum() if x.dtype == "object" else False)

In [ ]:
#checks for missing values
df.isnull().sum()

this part cleans the nan values and handles with it

In [ ]:
df.dropna(subset=['last_name'], inplace=True)
df[df['last_name'].isnull()]

In [ ]:
df[df['title'].isnull()]
df.loc[df['title'].isnull(), 'title'] = "M"

In [ ]:
df['title'].value_counts()

In [ ]:
df[df['title']=="Mr"]
df.loc[df['title']=="Mr", 'title'] = "M"

In [ ]:
df[df['address'].isnull()]

In [ ]:
df[df['city'].isnull()]
df.loc[df['city'].isnull(),"city"] = ["Paris", "Draguignan"]

In [ ]:
df[df['phone_number'].isnull()]
df.dropna(subset=['phone_number'], inplace=True)



In [ ]:
df[df['social_security_regime'].isnull()]
df.loc[df['social_security_regime'].isnull(),"social_security_regime"] = "Régime général"

In [ ]:
#handling missing birthday
df[df['birth_date'].isnull()]
median_birthdate = pd.to_datetime(df['birth_date'], errors='coerce').median()
df['birth_date'].fillna(median_birthdate, inplace=True)

In [ ]:
df[df['num_children'].isnull()]
df.loc[df['num_children'].isnull(),"num_children"] = 0

In [ ]:
df[df['medical_care'].isnull()]
df.dropna(subset=['medical_care'], inplace=True)


In [ ]:
df[df['optical'].isnull()]
df.loc[df['optical'].isnull(),"optical"] = 'ECO'
df.loc[df['dental'].isnull(),"dental"] = 'ECO'

In [ ]:
#checks for illogical values 
df[(df['num_children'] == 0) & (df['child1_birth_date'].notnull() | df['child2_birth_date'].notnull() | df['child3_birth_date'].notnull() | df['child4_birth_date'].notnull() | df['child5_birth_date'].notnull())]

This parts handles messy data

In [ ]:
df['social_security_regime'].value_counts()

In [ ]:
#handle messy data
df.loc[df['social_security_regime']=="General scheme","social_security_regime"] = 'Régime général'
df[df['social_security_regime']=="General scheme"]

In [ ]:
df[df['social_security_regime'].isin(['??????-??????', '??????????? ?????','General Diet','Esquema geral','Általános séma'])]
df.loc[df['social_security_regime'].isin(['??????-??????', '??????????? ?????','General Diet','Esquema geral','Általános séma']),"social_security_regime"] = 'Régime général'

In [ ]:
df['source'].value_counts()

In [ ]:
df[df['source']=="Proges"]
df.loc[df['source']=="Proges","source"] = 'PROGES'


In [ ]:
df['hospitalization'].value_counts()


In [ ]:
df[df['hospitalization']=='3']

In [ ]:
df.loc[df['hospitalization']=='3',"hospitalization"] = 'ELEVE'
df.loc[df['hospitalization']=='3',"optical"] = 'ELEVE'
df.loc[df['hospitalization']=='3',"dental"] = 'ELEVE'
df.loc[df['hospitalization']=='3',"medical_care"] = 'MOYEN'

df.loc[df['hospitalization']=='Medium.',"hospitalization"] = 'MOYEN'
df.loc[df['optical']=='Medium.',"optical"] = 'MOYEN'
df.loc[df['medical_care']=='High.',"medical_care"] = 'ELEVE'
df.loc[df['dental']=='High.',"dental"] = 'ELEVE'


In [ ]:
df['dental'].value_counts()
df[df['dental']=='3']
df.loc[df['dental']=='3',"medical_care"] = 'MOYEN'
df.loc[df['dental']=='3',"optical"] = 'ELEVE'
df.loc[df['dental']=='3',"dental"] = 'ELEVE'

In [ ]:
df['optical'].value_counts()
df['medical_care'].value_counts()

In [ ]:
df['last_name'].value_counts()


In [ ]:
#checks for wrong dtypes
df[~df['birth_date'].apply(lambda x: isinstance(x, str))]

In [ ]:
df[(~df['spouse_birth_date'].apply(lambda x: isinstance(x, str))) & (df['spouse_birth_date'].notna()) ]
df[(~df['child1_birth_date'].apply(lambda x: isinstance(x, str))) & (df['child1_birth_date'].notna()) ]
df[(~df['child2_birth_date'].apply(lambda x: isinstance(x, str))) & (df['child2_birth_date'].notna()) ]
df[(~df['child3_birth_date'].apply(lambda x: isinstance(x, str))) & (df['child3_birth_date'].notna()) ]
df[(~df['child4_birth_date'].apply(lambda x: isinstance(x, str))) & (df['child4_birth_date'].notna()) ]
df[(~df['child5_birth_date'].apply(lambda x: isinstance(x, str))) & (df['child5_birth_date'].notna()) ]
df[(~df['submission_date'].apply(lambda x: isinstance(x, str))) & (df['submission_date'].notna()) ]

This parts explores fake or unusable data and handles it

In [ ]:
df["email"].value_counts()

In [ ]:
#explores owner inputs for test inputs 
df[df["email"] == "jake.doe@example.com"]

In [ ]:
#decode html entities in the string
from html import unescape
string_cols = ['last_name', 'first_name', 'title', 'address', 'city', 'email',"zip_code","phone_number",'social_security_regime', 'birth_date', 'effective_date', 'spouse_birth_date', 'source', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date', 'submission_date', 'medical_care', 'hospitalization', 'optical', 'dental']
for col in string_cols:
    df[col] = df[col].apply(lambda x: unescape(str(x)) if pd.notna(x) else x)

#removes the anti slash remaining before apostrophe
for col in string_cols:
    df[col] = df[col].str.replace(r'\\\'', "'", regex=True)

In [ ]:
df[df["id"]==688]

In [ ]:
#detects fake leads / test inputs 
import re

#function to detect fake names
def is_random_name(s, min_length=3,):

    # Numbers only or mixed (e.g., 'Random123')
    # Regex pattern: keep letters (including accents), spaces, hyphens, apostrophes
    pattern = r"[^a-zA-ZÀ-ÖØ-öø-ÿ0-9 @\'-\.]"
    if re.search(pattern, s):
        return True
    
    s = s.strip().lower()
    if len(s) < min_length:
        return True
    # Repeated characters (e.g., 'daaaa')
    if len(set(s)) / len(s) < 0.4:
        return True
    # Keyboard sequences
    keyboard_patterns = ['t+e+s+t+', 'j+a+k+e+', 'd+o+e+']
    if any(re.search(pattern, s) for pattern in keyboard_patterns):
        return True
    return False

pd.set_option('display.max_rows', 10)  # Show all rows

df2 = df[(df['last_name'].apply(is_random_name)) | (df['first_name'].apply(is_random_name))]
df2.to_csv('probably_fake_prospects.csv', index=False)

#exluded ids from fake leads after manual reviews of the file "probably_fake_prospects.csv"
ids_keep = [101,945,3584,6173,6391,7710,9041,9763,10003,11125,11800,
            11944,13432,14155,14954,15435,16032,16144,16286,16515]


df = df.drop(df2[~df2['id'].isin(ids_keep)].index)


In [ ]:
pip install dnspython

In [ ]:
pip install tqdm

In [ ]:
import dns.resolver
from tqdm import tqdm
import difflib

#function to check for non existing domains 
#use caching to speed the dns lookup
    
domain_cache = {}
known_domains = [
    "gmail.com", "yahoo.fr", "orange.fr", "free.fr", 
    "sfr.fr", "laposte.net", "live.fr", "hotmail.fr"
]


def check_domain_exists(email):
    domain = email.split("@")[1].lower()
    closest = difflib.get_close_matches(domain, known_domains, n=1, cutoff=0.8)
    if closest : 
        domain = closest[0] 

    if domain in domain_cache:
        return domain_cache[domain]
    try:
        dns.resolver.resolve(domain, "MX")
        domain_cache[domain] = True
        return True
    except:
        domain_cache[domain] = False
        return False



#function to detect fake emails


def is_random_email(e, min_length=2):
    global i
    e=e.lower()

    disposable_domains = [
    "mailinator.com", "10minutemail.com", "tempmail.com",
    "guerrillamail.com", "trashmail.com", "fakeinbox.com",
    "yopmail.com", "yopmail.fr", "yopmail.net", "jetable.org"
]
    if any(e.endswith("@" + d) for d in disposable_domains):
        return True

    # checks for fake usernames with many ndigits in emails or very short emails
    local_part = e.split('@')[0]
    if len(local_part) < min_length:
        return True
    
    # checks for fake usernames with many repeated caracters
    if len(set(local_part)) / len(local_part) <= 0.3:
        return True
    
    #checks for owner/test submissions
    keyboard_patterns = ['t+e+s+t+', 'j+a+k+e+', 'd+o+e+']
    if any(re.search(pattern, e) for pattern in keyboard_patterns):
        return True
  
    return False

tqdm.pandas(desc="Checking emails")

df2 = df[df['email'].progress_apply(is_random_email)]
df2.to_csv('probably_fake_emails_prospects.csv', index=False)

#exluded ids from fake leads after manual reviews of the file "probably_fake_emails_prospects.csv"
ids_keep = [3584,12480,13170]


df = df.drop(df2[~df2['id'].isin(ids_keep)].index)


In [ ]:
df[df['email'].progress_apply(is_random_email)]

In [ ]:
df# Step 1: find emails that appear more than once
duplicate_emails = df['email'][df['email'].duplicated(keep=False)]

# Step 2: show all rows with those emails
df2 = df[df['email'].isin(duplicate_emails)]
df2.to_csv('test.csv', index=False)



In [ ]:
df_grouped = df.groupby('email').apply(lambda x: x).reset_index(drop=True)
df_grouped.to_csv('test.csv', index=False)

In [ ]:
#check for duplicate data 
df['submission_date'] = pd.to_datetime(df['submission_date'], errors='coerce')  # Ensure datetime
# Sort by email and dcr for consecutive check
df_sorted = df.sort_values(['email', 'submission_date']).reset_index(drop=True)
# Identify duplicate emails
df_sorted['is_duplicate'] = df_sorted['email'].duplicated(keep=False)
# Calculate time difference between consecutive rows for same email
df_sorted['submission_date_diff'] = (df_sorted.groupby('email')['submission_date'].diff().dt.total_seconds() / 3600)

# Find duplicates that are consecutive or nearly consecutive (e.g., <24 hours)
close_duplicates = df_sorted[df_sorted['is_duplicate'] & (df_sorted['submission_date_diff'].notnull()) & (df_sorted['submission_date_diff'] <= 72) ]
# Include the previous row for each close duplicate
close_duplicate_ids = close_duplicates.index
paired_rows = df_sorted.loc[close_duplicate_ids.union(close_duplicate_ids - 1)]  # Include previous rows
result = paired_rows[paired_rows['email'].isin(close_duplicates['email'])]

result_last = result.sort_values(['email', 'submission_date']).drop_duplicates(subset=['email'], keep='last')

result_last.to_csv('test.csv', index=False)


In [ ]:
rows_to_drop = result[~result['id'].isin(result_last["id"])]
df = df[~df['id'].isin(rows_to_drop['id'])]
df

 

In [ ]:
df['num_children'] = df['num_children'].astype('Int64')
df.info()

In [ ]:
for col in ['birth_date', 'effective_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    print(df[col].isnull().sum())

This part derive the columns of ages, cleans wrong values, imputes wrong values

In [ ]:
mask = df['birth_date'] == '1953-04-04 00:00:00'
df.loc[mask, 'birth_date'] = '04/04/1953'
df[df['birth_date'] == '04/04/1953']

In [ ]:
#makes a copy for later to inspect missing/ messy date values that will convert to NAT
df_copy = df.copy()

In [ ]:
#converts to date columns to datetime
for col in ['birth_date', 'effective_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    df[col] = pd.to_datetime(df_copy[col], format='%d/%m/%Y', errors='coerce', dayfirst=True)

In [ ]:
#derives ages columns

for col in ['birth_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    df[f'{col.replace("birth_date", "age_at_submission")}'] = (df['submission_date'] - df[col]).dt.days // 365
    df[f'{col.replace("birth_date", "current_age")}'] = (pd.to_datetime('2025-09-13') - df[col]).dt.days // 365
    

In [ ]:
#verify -999 is not assigned to any wrong values of ages to use it later as indicator for invalid ages or non existing children ages
cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']
invalid_rows = df[df[cols].eq(-999).any(axis=1)][['id', 'last_name', 'submission_date','effective_date','child1_birth_date', 'child2_birth_date', 'child3_birth_date', 
              'child4_birth_date', 'child5_birth_date','num_children'] + cols]
print(f"Rows with any child age == -999: {len(invalid_rows)}")
print(invalid_rows.head())

In [ ]:
#checks for prospects non plausible ages
#saves to excel file to manually review if the ages are intentional or typos : most of them are typos
df[(df['age_at_submission'] < 18) | (df['age_at_submission'] > 100)].to_excel("prospects_invalid_age.xlsx", index=False)
df = df[df['id'] != 5858]
#imputing the wrong values with the median of the column
valid_ages = df[(df['age_at_submission'] >= 18)]['age_at_submission']
median_age = valid_ages.median()
df.loc[(df['age_at_submission'] < 18), 'age_at_submission'] = median_age
df[(df['age_at_submission'] < 18)]

In [ ]:
#checks for spouses non plausible ages
#saves to excel file to manually review if the ages are intentional or typos
df[(df['spouse_age_at_submission'] < 18) | (df['spouse_age_at_submission'] > 100)].to_excel("spouses_invalid_age.xlsx", index=False)
#imputes the wrong values with the median of the column
valid_ages = df[(df['spouse_age_at_submission'] >= 18)]['spouse_age_at_submission']
median_age = valid_ages.median()
df.loc[(df['spouse_age_at_submission'] < 18), 'spouse_age_at_submission'] = median_age
df[(df['spouse_age_at_submission'] < 18)]


In [ ]:
#checks for children non plausible ages
#saves to excel file to manually review if the ages are intentional or typos
child_cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']
for col in child_cols:
    df[(df[col] > 25) & (df[col] < 60)][['id', 'first_name', 'last_name', 'email','age_at_submission','spouse_age_at_submission','submission_date',col.replace('age_at_submission', 'birth_date'), 'child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']].to_excel(f"{col.replace('age_at_submission','invalid_age.xlsx')}", index=False)
    # Set >25 to 0 (non-dependents)
    df.loc[(df[col] > 25) | (df[col] < 0) , col] = -999

# Update num_children based on valid ages (≤25)
df['child_count'] = df[child_cols].apply(lambda x: (x >= 0) & (x <= 25)).sum(axis=1)
df.loc[:, 'num_children'] = df['child_count']
df = df.drop(columns=['child_count'])

In [ ]:
#checks for plausible wrong ages 
df['spouse_age_diff'] = abs(df['age_at_submission'] - df['spouse_age_at_submission'])
print("Suspicious spouse age gaps (>30 years):")
pd.set_option('display.max_rows', 10)
print(df[(df['spouse_age_at_submission'] > 0) & (df['spouse_age_diff'] > 30)][['id', 'last_name', 'age_at_submission', 'spouse_age_at_submission', 'spouse_age_diff']])

In [ ]:
child_cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']
for col in child_cols:
    df['child_age_diff'] = df['age_at_submission'] - df[col]
    df[( (df["age_at_submission"] <= 85) | ((df["spouse_age_at_submission"] <= 70) & (df["spouse_age_at_submission"] != 0) ) ) 
   & (df[col] > 0) 
   & (df["child_age_diff"] < 15)][["id","title","email","age_at_submission","spouse_age_at_submission","num_children",col,"child_age_diff",'child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']].to_excel(col+"_vs_prospect_age.xlsx")
df[(df['age_at_submission'] < 25) & (df['num_children'] > 2)].to_excel("young_prospects_with_many_children.xlsx")

In [ ]:
#drop developer inserted test row
df = df.drop(index=df[df['id'] == 16386].index)

In [ ]:
df[df["child1_age_at_submission"] < 0]

In [ ]:
col = 'child1_birth_date'
mask = df[col].isna() & df_copy[col].notna() & (df_copy[col] != '')
print(df[mask][["id",col,col.replace("birth_date","age_at_submission")]])

In [ ]:
for col in ['birth_date', 'spouse_birth_date']:
    mask = df[col].isna() & df_copy[col].notna() & (df_copy[col] != '')
    df.loc[mask, col.replace("birth_date", "current_age")] = df[col.replace("birth_date", "current_age")].median()
    df.loc[mask, col.replace("birth_date", "age_at_submission")] = df[col.replace("birth_date", "age_at_submission")].median()


for col in ['birth_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    df.loc[df[col.replace("birth_date", "current_age")].isna(), col.replace("birth_date", "current_age")] = -999
    df.loc[df[col.replace("birth_date", "age_at_submission")].isna(), col.replace("birth_date", "age_at_submission")] = -999


# Check results
print(df[['id', 'first_name', 'age_at_submission', 'current_age', 
         'spouse_age_at_submission', 'spouse_current_age', 
         'child1_age_at_submission', 'child1_current_age']].head())

In [ ]:
# Update num_children based on valid ages (≤25)
df['child_count'] = df[child_cols].apply(lambda x: (x >= 0) & (x <= 25)).sum(axis=1)
df.loc[:, 'num_children'] = df['child_count']
df = df.drop(columns=['child_count'])

In [ ]:
# Verify no NaN in age columns
age_columns = ['age_at_submission', 'spouse_age_at_submission', 
               'child1_age_at_submission', 'child2_age_at_submission', 
               'child3_age_at_submission', 'child4_age_at_submission', 
               'child5_age_at_submission', 'current_age', 'spouse_current_age', 
               'child1_current_age', 'child2_current_age', 
               'child3_current_age', 'child4_current_age', 'child5_current_age']
print(df[age_columns].isnull().sum())  # Should be 0
# Convert to int64
for col in age_columns:
    df[col] = df[col].astype('int64')
# Verify
print(df[age_columns].dtypes)  # Should be int64
print(df[['id', 'first_name', 'age_at_submission', 'spouse_age_at_submission', 'child1_age_at_submission']].head())

In [ ]:
for col in ['birth_date', 'effective_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    print(df_copy[col].isnull().sum())

In [ ]:
for col in ['birth_date', 'effective_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    # Rows where original was non-null/empty but became NaT after conversion
    nat_rows = df[df[col].isna() & df_copy[col].notna() & (df_copy[col] != '')][['id', 'first_name', 'last_name', col,'medical_care','hospitalization','optical','dental']]
    print(f"NaT in {col}: {len(nat_rows)} rows")
    print(nat_rows)
    print(f"Original values: {df_copy.loc[nat_rows.index, col].values}")



In [ ]:
for col in ['birth_date', 'spouse_birth_date', 'child1_birth_date', 'child2_birth_date', 'child3_birth_date', 'child4_birth_date', 'child5_birth_date']:
    # Rows where original was non-null/empty but became NaT after conversion
    print(df[df[col].isna() & df_copy[col].notna() & (df_copy[col] != '')][col.replace("birth_date", "current_age")] )
    print(df[df[col].isna() & df_copy[col].notna() & (df_copy[col] != '')][col.replace("birth_date", "age_at_submission")])

Prepares for calculating the score of the prospect to detect hot leads

In [ ]:
#derives columns to use in the prioritization score
df['days_to_effective'] = (df['effective_date'] - df['submission_date'].dt.normalize()).dt.days

In [ ]:
df.loc[df['days_to_effective'].isna(), 'effective_date'] = df['submission_date'].dt.normalize()

In [ ]:
df['days_to_effective'] = (df['effective_date'] - df['submission_date'].dt.normalize()).dt.days
df.loc[df["days_to_effective"]<0,"days_to_effective"] = 0
df[df["days_to_effective"]<0]

In [ ]:
df['days_to_effective'] = df['days_to_effective'].astype('Int64')

In [ ]:
pip install openpyxl

In [ ]:
fake_patterns = ['0000000000', '0600000000', '0700000000', '0123456789','1234567890', '0606060606', '0707070707']
df[df['phone_number'].isin(fake_patterns)]


In [ ]:
child_cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']
for col in child_cols:
    df['child_age_diff'] = df['age_at_submission'] - df[col]
    df[( (df["age_at_submission"] <= 85) | ((df["spouse_age_at_submission"] <= 70) & (df["spouse_age_at_submission"] != 0) ) ) 
   & (df[col] > 0) 
   & (df["child_age_diff"] < 15)][["id","title","email","age_at_submission","spouse_age_at_submission","num_children",col,"child_age_diff",'child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']].to_excel(col+"_vs_prospect_age.xlsx")
df[(df['age_at_submission'] < 25) & (df['num_children'] > 2)].to_excel("young_prospects_with_many_children.xlsx")

In [ ]:
df[df["child1_age_at_submission"]!=-999]["child1_age_at_submission"].count()

In [ ]:
#verifies typos for illogical parent children ages gaps 
import numpy as np
# Define child columns
child_cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission', 
              'child4_age_at_submission', 'child5_age_at_submission']

# Calculate age gaps for prospect and spouse

#list f column names
list_gaps_prospect = []
list_gaps_spouse = []

#function

for col in child_cols:
    list_gaps_prospect.append(f'{col}_gap_prospect')
    list_gaps_spouse.append(f'{col}_gap_spouse')
# Initialize gap columns
    df[f'{col}_gap_prospect'] = -999
    df[f'{col}_gap_spouse'] = -999
    
    # Calculate gaps where child age is valid (not -999)
    valid_age_mask = df[col] != -999
    df.loc[valid_age_mask, f'{col}_gap_prospect'] = df['age_at_submission'] - df[col]
    df.loc[valid_age_mask & (df['spouse_age_at_submission'] > 0), f'{col}_gap_spouse'] = df['spouse_age_at_submission'] - df[col]


# Select rows with illogical gaps (>50 years for either prospect or spouse, valid child ages 0-25)
mask = False
for col in child_cols:
    mask |= ((df[col] >= 0) & (df[col] <= 25)) & \
            (((df[f'{col}_gap_prospect'] > 50) & \
             ((df['spouse_age_at_submission'] > 0) & (df[f'{col}_gap_spouse'] > 50)))  | ((df[f'{col}_gap_prospect'] > 50) & (df['spouse_age_at_submission'] == -999) ))
    
df['max_gap_prospect'] = df[list_gaps_prospect].max(axis=1)
df['max_gap_spouse'] = df[list_gaps_spouse].max(axis=1)

def min_positive(a, b):
    vals = [x for x in [a, b] if x > 0]  # keep only positives
    return min(vals) if vals else np.nan  # return min positive, NaN if none

df['min_parent_gap'] = df.apply(lambda row: min_positive(row['max_gap_prospect'], 
                                                         row['max_gap_spouse']), axis=1)



illogical_rows = df[mask][['id', 'last_name', 'title','age_at_submission', 'spouse_age_at_submission', 
                           'num_children',"max_gap_prospect",'max_gap_spouse','min_parent_gap']]

# Display results
print(f"Rows with illogical parent-child age gaps (>50 years): {len(illogical_rows)}")
pd.set_option('display.max_rows', 10)
print(illogical_rows[(illogical_rows["max_gap_spouse"]>45)&(illogical_rows["max_gap_prospect"]>60)])
print(f"\nRows with illogical prospect-child age gaps (>70 years)")
print(illogical_rows[(illogical_rows["max_gap_prospect"]>70)])


# Optional: Drop temporary gap columns
df = df.drop(columns=[f'{col}_gap_prospect' for col in child_cols] + 
                     [f'{col}_gap_spouse' for col in child_cols])
df = df.drop(columns=['max_gap_prospect','max_gap_spouse','min_parent_gap'])
print("check illogical values count")

test = illogical_rows[((illogical_rows["max_gap_spouse"]>45)&(illogical_rows["title"]=="M"))|((illogical_rows["max_gap_prospect"]>45)&(illogical_rows["title"]=="Mme"))]
print(test[(test["title"]=="M")&(test["max_gap_spouse"]>65)&(test["max_gap_prospect"]>65)]["max_gap_prospect"].count())

# Save results for review
illogical_rows.to_excel('illogical_child_age_gaps.xlsx', index=False)
illogical_rows[((illogical_rows["max_gap_spouse"]>70)&(illogical_rows["title"]=="Mme"))|((illogical_rows["max_gap_prospect"]>70)&(illogical_rows["title"]=="M"))].to_excel('illogical_father_child_age_gaps.xlsx', index=False)
illogical_rows[((illogical_rows["max_gap_spouse"]>45)&(illogical_rows["title"]=="M"))|((illogical_rows["max_gap_prospect"]>45)&(illogical_rows["title"]=="Mme"))].to_excel('illogical_mother_child_age_gaps.xlsx', index=False)
illogical_rows[((illogical_rows["max_gap_spouse"]>45)&(illogical_rows["title"]=="M"))|((illogical_rows["max_gap_prospect"]>70)&(illogical_rows["title"]=="M"))|((illogical_rows["max_gap_prospect"]>45)&(illogical_rows["title"]=="Mme"))|((illogical_rows["max_gap_spouse"]>70)&(illogical_rows["title"]=="Mme"))].to_excel('illogical_final_child_age_gaps.xlsx', index=False)

#Validated rare age gaps with research, keeping 73 rows as plausible adoptions/IVF/Step-parenting


In [ ]:
#check outliers 

def detect_outliers(col):
    # Exclude -999 (invalid ages)
    valid_ages = df[df[col] != -999][col]
    Q1 = valid_ages.quantile(0.25)
    Q3 = valid_ages.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    print("Q1 : ", Q1)
    print("Q3 : ", Q3)
    print("IQR : ", IQR)

    print("lower bound : ", lower_bound)
    print("upper bound : ", upper_bound)

    outliers = df[(df[col] != -999) & ((df[col] < lower_bound) | (df[col] > upper_bound))][['id', 'last_name', col]]
    print(f"Outliers in {col} (excluding -999): {len(outliers)} rows")
    print(outliers)
    return outliers

# Check prospect and spouse ages
pd.set_option('display.max_rows', None)


# Check children ages
child_cols = ['child1_age_at_submission', 'child2_age_at_submission', 'child3_age_at_submission',
              'child4_age_at_submission', 'child5_age_at_submission']

# Check children ages
for col in child_cols:
    detect_outliers(col).to_excel(col.replace("age_at_submission","ages_outliers.xlsx"))

In [ ]:
#check zipcode values validity
invalid_zip = df[~df['zip_code'].str.match(r'^\d{5}$')]
print(f"Invalid zip codes: {len(invalid_zip)}")
print(invalid_zip[['id', 'last_name', 'zip_code']])

In [ ]:
#saving the cleaned data after removing wrong rows or typos before encoding the data
df.to_csv('../data/pre_encoded_data.csv', index=False)

This part encodes the data and calculate the score

In [ ]:
df.info()

In [ ]:
level_map = {'ECO': 1, 'MOYEN': 2, 'ELEVE': 3, 'MAXI': 4}
df['medical_care'] = df['medical_care'].map(level_map)
df['hospitalization'] = df['hospitalization'].map(level_map)
df['optical'] = df['optical'].map(level_map)
df['dental'] = df['dental'].map(level_map)

In [ ]:
df['social_security_regime'].value_counts()

In [ ]:
regime_map = {
    'Régime général': 1,
    'Alsace-Moselle': 2,
    'Régime TNS': 3,
    'Régime agricole': 4,
    'Hors sécu': 5,
    'Régime CFE': 6
}
df['social_security_regime_encoded'] = df['social_security_regime'].map(regime_map)

In [ ]:
regime_weight_map = {
    'Régime général': 0.5,
    'Alsace-Moselle': 0,
    'Régime TNS': 1,
    'Régime agricole': 0.5,
    'Hors sécu': 0.8,
    'Régime CFE': 0.5
}
df['regime_score'] = df['social_security_regime'].map(regime_weight_map)

In [ ]:
pd.set_option('display.max_rows', None)
df[["source"]].value_counts()

In [ ]:
df['source_score'] = df['source'].apply(lambda x: 1 if x != 'web' else 0)

In [ ]:
urban_zipcodes = ['75', '77', '92', '93', '94', '95']  # Île-de-France
df['zip_score'] = df['zip_code'].str[:2].apply(lambda x: 1 if x in urban_zipcodes else 0)

In [ ]:
df['priority_score'] = (df['medical_care'] + df['hospitalization'] + df['optical'] + df['dental'] + \
                       df['num_children'].apply(lambda x: 2 if x > 1 else 0) + df['regime_score'] +\
                       df['age_at_submission'].apply(lambda x: 1 if x > 50 else 0) +\
                       df['spouse_age_at_submission'].apply(lambda x: 1 if x != 999 else 0) +\
                        df['days_to_effective'].apply(lambda x: 1 if 0 <= x <= 90 else 0) +\
                        df['source_score'])

In [ ]:
print(df['social_security_regime'].value_counts())  # Check sample sizes
print("Alsace-Moselle :\n", df[df['social_security_regime'] == 'Alsace-Moselle'][['medical_care', 'hospitalization', 'optical', 'dental']].mean())  # Check coverage
print("Régime TNS :\n", df[df['social_security_regime'] == 'Régime TNS'][['medical_care', 'hospitalization', 'optical', 'dental']].mean())

In [ ]:
df[['age_at_submission', 'medical_care', 'hospitalization', 'optical', 'dental']].corr()['age_at_submission']

In [ ]:
df.to_csv('../data/cleaned_prospects.csv', index=False)

In [ ]:
df.info()